# 12_model_baseline_comparison_canonical_260514

Canonical Step 12 fixed-parameter model baseline comparison. This notebook rebuilds Step 12c from canonical inputs only and excludes archived old Step 12 / Step 12r metrics from candidate selection.

In [1]:
from pathlib import Path
import os, sys, json, math, zipfile, importlib.util, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
try:
    from sklearn.model_selection import StratifiedGroupKFold
except Exception as exc:
    raise RuntimeError('StratifiedGroupKFold is required; ordinary KFold is not allowed.') from exc

STEP_NAME = '12_model_baseline_comparison_canonical_260514'
PREFIX = '12c'
EXPECTED_ROOTS = {'C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction'}
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents)
REPO_ROOT = next((p for p in root_candidates if (p / 'park.ingyeom').exists()), cwd)
if REPO_ROOT.as_posix() != 'C:/Code/ott-churn-prediction':
    raise RuntimeError(f'Repo root mismatch: {REPO_ROOT}')
PARK = REPO_ROOT / 'park.ingyeom'
NOTE_PATH = PARK / 'note.md'
NB_PATH = PARK / 'notebook' / STEP_NAME / f'{STEP_NAME}.ipynb'

BASE_MODEL = PARK / 'reports' / 'models' / STEP_NAME
BASE_FIG = PARK / 'reports' / 'figures' / STEP_NAME
def choose_output_folder(base):
    if base.exists() and any(base.iterdir()):
        run = base / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
        run.mkdir(parents=True, exist_ok=False)
        return run
    base.mkdir(parents=True, exist_ok=True)
    return base
MODEL_OUT = choose_output_folder(BASE_MODEL)
FIG_OUT = choose_output_folder(BASE_FIG)
ZIP_DIR = PARK / 'zip'
ZIP_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'

def pstr(p): return str(Path(p).resolve())
def read_csv(path): return pd.read_csv(path, low_memory=False)
def write_csv(df, name):
    path = MODEL_OUT / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path
def pass_all(path, status_col='status'):
    df = read_csv(path)
    if status_col not in df.columns:
        return False
    return df[status_col].astype(str).str.upper().eq('PASS').all()
def latest_valid_folder(base, required, final_name):
    candidates = []
    if base.exists():
        candidates.append(base)
        candidates.extend([p for p in base.glob('run_*') if p.is_dir()])
    valid = []
    for folder in candidates:
        files_ok = all((folder / r).exists() for r in required)
        checks_ok = files_ok and pass_all(folder / final_name)
        if files_ok and checks_ok:
            valid.append(folder)
    if not valid:
        raise RuntimeError(f'No valid folder found under {base}')
    return sorted(valid, key=lambda x: x.stat().st_mtime, reverse=True)[0]

paths = {
    'primary_table': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_conservative_features.csv',
    'primary_index': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_index.csv',
    'role_dict': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_column_role_dictionary.csv',
    'safe_cols': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_conservative_safe_candidate_columns.csv',
    'review_cols': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_review_required_columns.csv',
    'forbidden_cols': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_forbidden_drop_columns.csv',
    'mapping_07': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_mapping_conservative_features.csv',
}
folder_09b = PARK / 'reports' / 'audits' / '09b_raw_view_window_validation_260514' / 'run_20260514_130402'
req_09b = ['09b_final_checks.csv','09b_core_usage_recalculation_comparison.csv','09b_window_validation_decision.csv']
folder_10_base = PARK / 'reports' / 'eda' / '10_feature_eda_260513'
req_10 = ['10_final_checks.csv','10_feature_eda_catalog.csv','10_focus_feature_deep_dive_summary.csv','10_handoff_to_11_and_17.csv','10_open_risks_for_next_steps.csv']
folder_11b_base = PARK / 'reports' / 'models' / '11b_baseline_growth_history_ladder_fix_260514'
req_11b = ['11b_final_checks.csv','11b_modeling_input_contract.csv','11b_feature_ladder_definition.csv','11b_ladder_contamination_check.csv','11b_dataset_scope_definition.csv','11b_model_registry.csv','11b_cv_summary_metrics.csv','11b_best_baseline_by_scope.csv','11b_ladder_growth_summary.csv','11b_train_valid_gap_audit.csv','11b_score_orientation_policy.csv','11b_open_risks_for_next_steps.csv','11b_handoff_to_12_model_comparison.csv']
folder_sem = PARK / 'reports' / 'audits' / '11b_semantic_validation_and_interpretation_patch_260514'
req_sem = ['11b_semantic_final_checks.csv','11b_canonical_status_decision.csv','11b_ladder_semantic_classification.csv','11b_ladder_interpretation_guardrail.csv','11b_handoff_to_12_semantic_requirements.csv']
archive_root = PARK / '_archive'

all_required = list(paths.values()) + [folder_09b / x for x in req_09b] + [folder_sem / x for x in req_sem]
missing = [pstr(p) for p in all_required if not p.exists()]
if missing:
    raise RuntimeError('Missing required inputs: ' + json.dumps(missing, ensure_ascii=False))
if not pass_all(folder_09b / '09b_final_checks.csv'):
    raise RuntimeError('09b final checks are not all PASS.')
folder_10 = latest_valid_folder(folder_10_base, req_10, '10_final_checks.csv')
folder_11b = latest_valid_folder(folder_11b_base, req_11b, '11b_final_checks.csv')
if not pass_all(folder_sem / '11b_semantic_final_checks.csv'):
    raise RuntimeError('11b semantic patch final checks are not all PASS.')

df = read_csv(paths['primary_table'])
safe_df = read_csv(paths['safe_cols'])
review_df = read_csv(paths['review_cols'])
forbid_df = read_csv(paths['forbidden_cols'])
ladder_df = read_csv(folder_11b / '11b_feature_ladder_definition.csv')
baseline_11b = read_csv(folder_11b / '11b_best_baseline_by_scope.csv')

def col_list(frame):
    for c in ['column_name','feature_name','column']:
        if c in frame.columns:
            return frame[c].dropna().astype(str).tolist()
    return []
review_cols = set(col_list(review_df))
forbidden_cols = set(col_list(forbid_df))
safe_cols = set(col_list(safe_df))
metadata_cols = {'USER_KEY','source_row_number','is_repurchase','repurchase_score','churn_risk'}

def parse_features(s):
    if pd.isna(s) or str(s).strip() == '': return []
    return [x.strip() for x in str(s).split(';') if x.strip()]
def ladder_features(scope, want_l5=False):
    sub = ladder_df[ladder_df['dataset_scope'].astype(str).eq(scope)].copy()
    if sub.empty:
        sub = ladder_df[ladder_df['dataset_scope'].astype(str).str.contains('overall', na=False)].copy()
    if want_l5:
        pick = sub[sub['ladder_step'].astype(str).str.contains('L5', na=False)]
    else:
        pick = sub[sub['ladder_step'].astype(str).str.contains('L4', na=False)]
    if pick.empty:
        pick = sub.sort_values('ladder_order').tail(1)
    feats = parse_features(pick.iloc[-1]['feature_names'])
    return feats

scopes = ['overall_without_promotion','overall_with_promotion','promotion_only','nonpromotion_only']
scope_filters = {
    'overall_without_promotion': pd.Series(True, index=df.index),
    'overall_with_promotion': pd.Series(True, index=df.index),
    'promotion_only': df['is_promotion'].eq(1),
    'nonpromotion_only': df['is_promotion'].eq(0),
}
feature_by_scope = {}
for scope in scopes:
    feats = ladder_features(scope, want_l5=(scope == 'overall_with_promotion'))
    if not feats:
        feats = sorted([c for c in safe_cols if c in df.columns])
    feats = [f for f in feats if f in df.columns and f not in metadata_cols and f not in review_cols and f not in forbidden_cols]
    if scope != 'overall_with_promotion':
        feats = [f for f in feats if f != 'is_promotion']
    elif 'is_promotion' not in feats:
        feats = feats + ['is_promotion']
    feature_by_scope[scope] = feats

required_columns = {'USER_KEY','is_repurchase','is_promotion'}
if not required_columns.issubset(df.columns):
    raise RuntimeError('Required columns missing from modeling table.')
if len(feature_by_scope['overall_without_promotion']) != 22:
    raise RuntimeError(f"Expected 22 conservative behavior features, found {len(feature_by_scope['overall_without_promotion'])}.")

font_candidates = ['Malgun Gothic','Noto Sans CJK KR','Noto Sans KR','NanumGothic','AppleGothic']
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
font_used = next((f for f in font_candidates if f in available_fonts), None)
if font_used:
    plt.rcParams['font.family'] = font_used
plt.rcParams['axes.unicode_minus'] = False
viz_warnings = []
if not font_used:
    viz_warnings.append({'warning_type':'korean_font','status':'WARNING','message':'Preferred Korean font not found; matplotlib default font used.'})

model_specs = []
def add_model(name, required, available, builder, params, reason=''):
    model_specs.append({'model_name':name,'required_or_optional':required,'import_available':available,'builder':builder,'fixed_parameters':params,'unavailable_reason':reason})
add_model('LogisticRegression','required',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler()),('model',LogisticRegression(max_iter=2000, solver='lbfgs'))]),'SimpleImputer median; StandardScaler; LogisticRegression(max_iter=2000, solver=lbfgs)')
add_model('HistGradientBoosting','required',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(random_state=42))]),'SimpleImputer median; HistGradientBoostingClassifier(random_state=42)')
add_model('RandomForest','required',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]),'SimpleImputer median; RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42)')
add_model('GradientBoosting','required',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',GradientBoostingClassifier(random_state=42))]),'SimpleImputer median; GradientBoostingClassifier(random_state=42)')
add_model('ExtraTrees','required',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',ExtraTreesClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42))]),'SimpleImputer median; ExtraTreesClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,n_jobs=-1,random_state=42)')
if importlib.util.find_spec('lightgbm'):
    from lightgbm import LGBMClassifier
    add_model('LightGBM','optional',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',LGBMClassifier(n_estimators=300,learning_rate=0.05,num_leaves=31,subsample=0.9,colsample_bytree=0.9,random_state=42,n_jobs=-1,verbose=-1))]),'LGBMClassifier fixed 300 trees, lr=0.05')
else:
    add_model('LightGBM','optional',False,None,'LGBMClassifier fixed 300 trees, lr=0.05','module not installed')
if importlib.util.find_spec('xgboost'):
    from xgboost import XGBClassifier
    add_model('XGBoost','optional',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',XGBClassifier(n_estimators=300,learning_rate=0.05,max_depth=4,subsample=0.9,colsample_bytree=0.9,eval_metric='logloss',random_state=42,n_jobs=-1))]),'XGBClassifier fixed 300 trees, lr=0.05, max_depth=4')
else:
    add_model('XGBoost','optional',False,None,'XGBClassifier fixed 300 trees, lr=0.05, max_depth=4','module not installed')
if importlib.util.find_spec('catboost'):
    from catboost import CatBoostClassifier
    add_model('CatBoost','optional',True,lambda: Pipeline([('imputer',SimpleImputer(strategy='median')),('model',CatBoostClassifier(iterations=300,learning_rate=0.05,depth=4,random_seed=42,verbose=False))]),'CatBoostClassifier(iterations=300, lr=0.05, depth=4)')
else:
    add_model('CatBoost','optional',False,None,'CatBoostClassifier(iterations=300, lr=0.05, depth=4)','module not installed')
availability = pd.DataFrame([{k:v for k,v in m.items() if k != 'builder'} for m in model_specs])
availability['will_run'] = availability['import_available'].map(lambda x: 'yes' if x else 'no')
availability['tuning_performed'] = 'no'

fold_rows, split_rows, warning_rows = [], [], []
summary_rows, operating_rows, decile_rows, oof_rows = [], [], [], []
for scope in scopes:
    sdf = df.loc[scope_filters[scope]].copy().reset_index(drop=True)
    y = sdf['is_repurchase'].astype(int).to_numpy()
    groups = sdf['USER_KEY'].to_numpy()
    X = sdf[feature_by_scope[scope]].apply(pd.to_numeric, errors='coerce')
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    splits = list(cv.split(X, y, groups))
    for fold_id, (tr, va) in enumerate(splits, 1):
        train_groups = set(groups[tr]); valid_groups = set(groups[va])
        overlap = len(train_groups.intersection(valid_groups))
        valid_classes = sorted(pd.Series(y[va]).unique().tolist())
        split_rows.append({'dataset_scope':scope,'fold_id':fold_id,'train_rows':len(tr),'valid_rows':len(va),'train_group_count':len(train_groups),'valid_group_count':len(valid_groups),'group_overlap_count':overlap,'valid_class_count':len(valid_classes),'valid_classes':';'.join(map(str,valid_classes)),'status':'PASS' if overlap == 0 and len(valid_classes) == 2 else 'FAIL'})
    for spec in model_specs:
        name = spec['model_name']
        if not spec['import_available']:
            warning_rows.append({'dataset_scope':scope,'model_name':name,'warning_type':'model_unavailable','severity':'WARNING','message':spec['unavailable_reason']})
            continue
        oof = np.full(len(sdf), np.nan)
        fold_auc, train_auc, aps, briers = [], [], [], []
        ok = True
        for fold_id, (tr, va) in enumerate(splits, 1):
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[va])) < 2:
                ok = False
                fold_rows.append({'dataset_scope':scope,'model_name':name,'fold_id':fold_id,'status':'FAIL','reason':'fold lacks both classes'})
                continue
            model = spec['builder']()
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                model.fit(X.iloc[tr], y[tr])
                for w in caught:
                    warning_rows.append({'dataset_scope':scope,'model_name':name,'warning_type':w.category.__name__,'severity':'WARNING','message':str(w.message)[:500]})
            p_va = model.predict_proba(X.iloc[va])[:,1]
            p_tr = model.predict_proba(X.iloc[tr])[:,1]
            oof[va] = p_va
            fa = roc_auc_score(y[va], p_va); ta = roc_auc_score(y[tr], p_tr)
            ap = average_precision_score(y[va], p_va); br = brier_score_loss(y[va], p_va)
            fold_auc.append(fa); train_auc.append(ta); aps.append(ap); briers.append(br)
            fold_rows.append({'dataset_scope':scope,'model_name':name,'fold_id':fold_id,'status':'PASS','train_auc':ta,'validation_auc':fa,'train_valid_gap':ta-fa,'average_precision':ap,'brier_score_loss':br,'train_rows':len(tr),'valid_rows':len(va)})
        if not ok or np.isnan(oof).any():
            continue
        oof_auc = roc_auc_score(y, oof); oof_ap = average_precision_score(y, oof); oof_brier = brier_score_loss(y, oof)
        gap_arr = np.array(train_auc) - np.array(fold_auc)
        summary_rows.append({'dataset_scope':scope,'model_name':name,'row_count':len(sdf),'feature_count':len(feature_by_scope[scope]),'oof_auc':oof_auc,'oof_average_precision':oof_ap,'oof_brier_score_loss':oof_brier,'mean_valid_auc':float(np.mean(fold_auc)),'std_valid_auc':float(np.std(fold_auc, ddof=1)),'min_valid_auc':float(np.min(fold_auc)),'max_valid_auc':float(np.max(fold_auc)),'mean_train_auc':float(np.mean(train_auc)),'mean_train_valid_gap':float(np.mean(gap_arr)),'max_train_valid_gap':float(np.max(gap_arr)),'fixed_parameter_comparison':'yes','tuning_performed':'no'})
        result = sdf[['source_row_number','USER_KEY','is_promotion','is_repurchase']].copy() if 'source_row_number' in sdf.columns else sdf[['USER_KEY','is_promotion','is_repurchase']].copy()
        result['dataset_scope'] = scope; result['model_name'] = name; result['repurchase_score'] = oof; result['churn_risk'] = 1 - oof
        oof_rows.append(result)
        total_nonrep = int((y == 0).sum()); base_nr = total_nonrep / len(y)
        ranked = result.sort_values('churn_risk', ascending=False).reset_index(drop=True)
        for kfrac in [0.10, 0.20]:
            k = max(1, int(math.ceil(len(ranked) * kfrac)))
            top = ranked.head(k)
            top_nr = int((top['is_repurchase'] == 0).sum())
            prec = top_nr / k; rec = top_nr / total_nonrep if total_nonrep else np.nan; lift = prec / base_nr if base_nr else np.nan
            operating_rows.append({'dataset_scope':scope,'model_name':name,'ranking_score_used':'churn_risk','event_of_interest':'nonrepurchase','k_fraction':kfrac,'top_k_row_count':k,'top_k_nonrepurchase_count':top_nr,'precision_at_k':prec,'recall_at_k':rec,'lift_at_k':lift,'baseline_nonrepurchase_rate':base_nr,'observed_nonrepurchase_rate_in_top_k':prec})
        for score_type, score_col, event_col, event_def in [('repurchase_score','repurchase_score','is_repurchase','repurchase = is_repurchase=1'),('churn_risk','churn_risk','nonrepurchase','nonrepurchase = is_repurchase=0')]:
            tmp = result.copy()
            tmp['event'] = tmp['is_repurchase'] if event_col == 'is_repurchase' else 1 - tmp['is_repurchase']
            tmp = tmp.sort_values(score_col, ascending=False).reset_index(drop=True)
            tmp['decile'] = pd.qcut(np.arange(len(tmp)), 10, labels=False) + 1
            for dec, g in tmp.groupby('decile'):
                decile_rows.append({'dataset_scope':scope,'model_name':name,'score_type':score_type,'decile':int(dec),'row_count':len(g),'mean_score':g[score_col].mean(),'observed_event_rate':g['event'].mean(),'event_definition':event_def,'caution':'descriptive diagnostic only; not deployment calibration claim'})

fold_metrics = pd.DataFrame(fold_rows)
summary = pd.DataFrame(summary_rows)
operating = pd.DataFrame(operating_rows)
deciles = pd.DataFrame(decile_rows)
all_oof = pd.concat(oof_rows, ignore_index=True) if oof_rows else pd.DataFrame()

policy_rows = [
    {'candidate_type':'highest_auc_candidate','rule':'model with highest OOF AUC within dataset scope'},
    {'candidate_type':'operating_metric_candidate','rule':'highest lift@top10% using churn_risk descending; tie precision@top10%, then OOF AUC'},
    {'candidate_type':'stability_aware_candidate','rule':'pool models within 0.005 OOF AUC of best; if empty use 0.010; choose lowest mean_train_valid_gap; tie lowest std_valid_auc, then highest OOF AUC'},
    {'candidate_type':'interpretation_candidate','rule':'prefer stable lower-gap simpler model if AUC within 0.010 of best; otherwise record review'},
    {'candidate_type':'recommended_candidate_for_14','rule':'candidate to tune only if worthwhile; avoid severe overfit; not chosen only because highest AUC'},
    {'candidate_type':'recommended_candidate_for_16','rule':'candidate to interpret later with SHAP; stable enough and tree/linear family; may differ from Optuna candidate'},
]
simplicity_rank = {'LogisticRegression':1,'RandomForest':2,'ExtraTrees':2,'GradientBoosting':3,'HistGradientBoosting':3,'LightGBM':4,'XGBoost':4,'CatBoost':4}
cand_rows, stab_rows = [], []
for scope in scopes:
    ss = summary[summary['dataset_scope'].eq(scope)].copy()
    oo = operating[(operating['dataset_scope'].eq(scope)) & (operating['k_fraction'].eq(0.10))].copy()
    best_auc = ss['oof_auc'].max()
    high = ss.sort_values(['oof_auc','mean_train_valid_gap'], ascending=[False, True]).iloc[0]
    oo = oo.merge(ss[['dataset_scope','model_name','oof_auc']], on=['dataset_scope','model_name'], how='left')
    op = oo.sort_values(['lift_at_k','precision_at_k','oof_auc'], ascending=[False, False, False]).iloc[0]
    pool = ss[ss['oof_auc'] >= best_auc - 0.005].copy(); tol = 0.005
    if pool.empty:
        pool = ss[ss['oof_auc'] >= best_auc - 0.010].copy(); tol = 0.010
    st = pool.sort_values(['mean_train_valid_gap','std_valid_auc','oof_auc'], ascending=[True, True, False]).iloc[0]
    interp_pool = ss[ss['oof_auc'] >= best_auc - 0.010].copy()
    interp_pool['simplicity'] = interp_pool['model_name'].map(simplicity_rank).fillna(9)
    interp = interp_pool.sort_values(['mean_train_valid_gap','simplicity','std_valid_auc','oof_auc'], ascending=[True, True, True, False]).iloc[0]
    tune_pool = ss[ss['mean_train_valid_gap'] <= max(0.03, ss['mean_train_valid_gap'].median() + ss['mean_train_valid_gap'].std(ddof=0))].copy()
    if tune_pool.empty: tune_pool = pool.copy()
    tune = tune_pool.sort_values(['oof_auc','mean_train_valid_gap'], ascending=[False, True]).iloc[0]
    shap = interp
    caution = 'fixed-parameter comparison only; not final model, threshold, segmentation, or causal evidence'
    cand_rows.append({'dataset_scope':scope,'highest_auc_candidate':high['model_name'],'highest_auc_oof_auc':high['oof_auc'],'operating_metric_candidate':op['model_name'],'operating_metric_lift10':op['lift_at_k'],'stability_aware_candidate':st['model_name'],'stability_aware_gap':st['mean_train_valid_gap'],'interpretation_candidate':interp['model_name'],'recommended_candidate_for_14':tune['model_name'],'recommended_candidate_for_16':shap['model_name'],'reason':'rules applied from 12c policy using OOF AUC, lift@10, train-valid gap, fold stability, and simplicity','caution':caution})
    stab_rows.append({'dataset_scope':scope,'best_oof_auc':best_auc,'candidate_pool_auc_tolerance_used':tol,'candidate_pool_models':';'.join(pool['model_name'].tolist()),'chosen_stability_model':st['model_name'],'chosen_gap':st['mean_train_valid_gap'],'chosen_std_valid_auc':st['std_valid_auc'],'why_not_automatically_highest_auc':'stability candidate is selected by gap and fold stability inside near-best AUC pool, not forced to highest AUC or any specific model','caution':caution})
candidate_df = pd.DataFrame(cand_rows)
stability_df = pd.DataFrame(stab_rows)
selected_models = set(candidate_df[['highest_auc_candidate','operating_metric_candidate','stability_aware_candidate','interpretation_candidate','recommended_candidate_for_14','recommended_candidate_for_16']].stack().dropna().tolist())
selected_oof = all_oof[all_oof['model_name'].isin(selected_models)].copy()
manifest = selected_oof.groupby(['dataset_scope','model_name']).size().reset_index(name='oof_row_count')
manifest['score_orientation'] = 'repurchase_score=P(is_repurchase=1); churn_risk=1-repurchase_score; top-k ranks churn_risk descending'

summary_for_vs = summary.rename(columns={'oof_auc':'oof_auc_12c'})
vs = summary_for_vs.merge(baseline_11b, on='dataset_scope', how='left', suffixes=('', '_11b'))
auc_11b_col = next((c for c in ['best_oof_auc','oof_auc_11b','mean_valid_auc_11b','mean_valid_auc'] if c in vs.columns), None)
if auc_11b_col:
    vs['delta_auc_vs_11b'] = vs['oof_auc_12c'] - pd.to_numeric(vs[auc_11b_col], errors='coerce')
else:
    vs['delta_auc_vs_11b'] = np.nan
vs['comparison_caution'] = '11b is prior ladder baseline reference; 12c is fixed model-family comparison, not final model selection'

gap_audit = summary[['dataset_scope','model_name','mean_train_auc','mean_valid_auc','mean_train_valid_gap','max_train_valid_gap']].copy()
gap_audit['status'] = np.where(gap_audit['mean_train_valid_gap'] > 0.05, 'WARNING', 'PASS')
gap_audit['caution'] = 'larger train-valid gap indicates possible overfit; avoid tuning candidate based only on AUC'
fold_stab = summary[['dataset_scope','model_name','std_valid_auc','min_valid_auc','max_valid_auc','oof_auc']].copy()
fold_stab['status'] = np.where(fold_stab['std_valid_auc'] > 0.03, 'WARNING', 'PASS')
fold_stab['caution'] = 'fold stability is diagnostic under group-aware CV'
archive_audit = pd.DataFrame([
    {'item':'old Step 12','status':'archived/deprecated','archive_root_detected':archive_root.exists(),'metrics_used_in_12c':'no','reason_excluded':'AUC-centered, lacked operating metrics','current_canonical_step':STEP_NAME},
    {'item':'old Step 12r','status':'archived/deprecated','archive_root_detected':archive_root.exists(),'metrics_used_in_12c':'no','reason_excluded':'candidate-selection logic issue','current_canonical_step':STEP_NAME},
])
scope_def = pd.DataFrame([{'dataset_scope':s,'row_filter':('all rows' if 'overall' in s else 'is_promotion=1' if s=='promotion_only' else 'is_promotion=0'),'target':'is_repurchase','positive_class':'is_repurchase=1 means repurchase','group_key':'USER_KEY','analysis_unit':'row-level / subscription-event-level'} for s in scopes])
feature_set_df = pd.DataFrame([{'dataset_scope':s,'feature_count':len(fs),'features':';'.join(fs),'uses_is_promotion_feature':'yes' if 'is_promotion' in fs else 'no'} for s,fs in feature_by_scope.items()])
contract = pd.DataFrame([
    {'contract_item':'analysis_unit','value':'row-level / subscription-event-level'},
    {'contract_item':'target','value':'is_repurchase'},
    {'contract_item':'positive_class','value':'is_repurchase=1 means repurchase'},
    {'contract_item':'score_orientation','value':'repurchase_score=P(is_repurchase=1); churn_risk=1-repurchase_score'},
    {'contract_item':'group_key','value':'USER_KEY only for CV grouping, never feature'},
    {'contract_item':'review_columns','value':'excluded'},
    {'contract_item':'forbidden_drop_columns','value':'excluded'},
    {'contract_item':'modeling_scope','value':'fixed-parameter family comparison only; no tuning, SHAP, thresholding, segmentation, or causality'},
])
safe_wording = pd.DataFrame([
    {'unsafe':'XGBoost가 가장 좋으니 최종 모델이다.','safer':'Step 12c의 고정 파라미터 비교에서 XGBoost가 높은 AUC 또는 top-k 지표를 보일 수 있으나, 최종 모델 여부는 안정성, 해석 가능성, SHAP, 튜닝 전후 비교 후 결정한다.'},
    {'unsafe':'AUC가 올라갔으니 캠페인 효과가 입증됐다.','safer':'AUC 상승은 예측 성능 개선이며, 마케팅 효과나 인과효과를 의미하지 않는다.'},
    {'unsafe':'top10 churn_risk가 캠페인 타겟이다.','safer':'top10 churn_risk는 운영 지표 진단용 구간이며, 실제 캠페인 기준은 세그먼트 설계와 실험 설계 후 정한다.'},
    {'unsafe':'lift@10이 높으니 바로 실행하면 된다.','safer':'lift@10은 후보 모델의 위험군 집중도를 보는 진단 지표이며, 실제 uplift는 A/B test가 필요하다.'},
    {'unsafe':'review 컬럼을 안 넣어도 정보 손실이 없다.','safer':'보수 baseline에서는 review 컬럼을 제외했으며, 정보 손실 가능성은 후속 sensitivity에서 검토한다.'},
])
risk_rows = [
    {'risk':'Fixed-parameter ranking may change after Optuna tuning.','severity':'medium','carry_forward':'Compare tuned and untuned candidates before final model selection.'},
    {'risk':'Top-k churn-risk metrics are diagnostics, not campaign thresholds.','severity':'high','carry_forward':'Do not use top-k as execution rule without segment and experiment design.'},
    {'risk':'Calibration deciles are descriptive and not deployment calibration evidence.','severity':'medium','carry_forward':'Revisit calibration after candidate finalization.'},
    {'risk':'Review columns remain excluded; potential information loss is untested.','severity':'medium','carry_forward':'Optional sensitivity can test review columns later under policy.'},
]
handoff14 = candidate_df[['dataset_scope','recommended_candidate_for_14','reason','caution']].rename(columns={'recommended_candidate_for_14':'candidate_for_optuna'})
handoff14['next_step'] = '14_optuna_candidate_tuning_260513 if tuning is needed; no tuning performed in 12c'
handoff16 = candidate_df[['dataset_scope','recommended_candidate_for_16','reason','caution']].rename(columns={'recommended_candidate_for_16':'candidate_for_shap'})
handoff16['next_step'] = '16_SHAP after candidate is stable enough for interpretation; no SHAP performed in 12c'

preflight = pd.DataFrame([
    {'check_name':'repo_root_checked','status':'PASS','value':REPO_ROOT.as_posix(),'notes':'absolute preflight passed'},
    {'check_name':'repo_root_matches_expected','status':'PASS','value':REPO_ROOT.as_posix(),'notes':'matches C:/Code/ott-churn-prediction'},
    {'check_name':'all_required_input_files_exist','status':'PASS','value':len(all_required),'notes':'required canonical inputs found'},
    {'check_name':'detected_09b_output_folder','status':'PASS','value':pstr(folder_09b),'notes':'explicit required folder used'},
    {'check_name':'detected_10_output_folder','status':'PASS','value':pstr(folder_10),'notes':'latest valid 10 folder'},
    {'check_name':'detected_11b_model_output_folder','status':'PASS','value':pstr(folder_11b),'notes':'latest valid 11b folder'},
    {'check_name':'detected_11b_semantic_patch_folder','status':'PASS','value':pstr(folder_sem),'notes':'semantic final checks all PASS'},
    {'check_name':'actual_model_output_folder','status':'PASS','value':pstr(MODEL_OUT),'notes':'12c write folder'},
    {'check_name':'actual_figure_output_folder','status':'PASS','value':pstr(FIG_OUT),'notes':'12c figure folder'},
])
preflight['actual_model_output_folder'] = pstr(MODEL_OUT); preflight['actual_figure_output_folder'] = pstr(FIG_OUT)

csv_outputs = {
 '12c_preflight_input_validation.csv': preflight,
 '12c_archived_old_12_exclusion_audit.csv': archive_audit,
 '12c_modeling_input_contract.csv': contract,
 '12c_model_availability.csv': availability.drop(columns=['unavailable_reason']).assign(unavailable_reason=availability['unavailable_reason']),
 '12c_dataset_scope_definition.csv': scope_def,
 '12c_feature_set_by_scope.csv': feature_set_df,
 '12c_cv_split_audit.csv': pd.DataFrame(split_rows),
 '12c_model_comparison_fold_metrics.csv': fold_metrics,
 '12c_model_comparison_summary.csv': summary,
 '12c_operating_metrics_at_k.csv': operating,
 '12c_calibration_decile_summary.csv': deciles,
 '12c_vs_11b_baseline_comparison.csv': vs,
 '12c_candidate_selection_policy.csv': pd.DataFrame(policy_rows),
 '12c_candidate_selection_by_scope.csv': candidate_df,
 '12c_stability_aware_candidate_by_scope.csv': stability_df,
 '12c_train_valid_gap_audit.csv': gap_audit,
 '12c_fold_stability_audit.csv': fold_stab,
 '12c_oof_predictions_selected_candidates.csv': selected_oof,
 '12c_oof_prediction_manifest.csv': manifest,
 '12c_modeling_warnings.csv': pd.DataFrame(warning_rows) if warning_rows else pd.DataFrame([{'dataset_scope':'all','model_name':'all','warning_type':'none','severity':'PASS','message':'No model warnings captured.'}]),
 '12c_safe_unsafe_wording.csv': safe_wording,
 '12c_open_risks_for_next_steps.csv': pd.DataFrame(risk_rows),
 '12c_handoff_to_14_optuna_candidate_tuning.csv': handoff14,
 '12c_handoff_to_16_shap_candidate_interpretation.csv': handoff16,
}
for name, frame in csv_outputs.items():
    write_csv(frame, name)

def savefig(name, title, fig):
    path = FIG_OUT / name
    fig.suptitle(title, fontsize=13)
    fig.text(0.5, 0.01, '고정 파라미터 baseline 비교이며 최종 모델, 캠페인 기준, 인과효과 주장이 아닙니다.', ha='center', fontsize=8)
    fig.tight_layout(rect=[0,0.04,1,0.94])
    fig.savefig(path, dpi=170)
    plt.close(fig)
    return path
fig_paths = []
pivot = summary.pivot(index='model_name', columns='dataset_scope', values='oof_auc')
fig, ax = plt.subplots(figsize=(10,6)); pivot.plot(kind='bar', ax=ax); ax.set_ylabel('OOF AUC'); ax.set_xlabel('모델'); ax.legend(fontsize=8); fig_paths.append(savefig('12c_fig_01_model_auc_by_scope.png','범위별 모델 OOF AUC 비교',fig))
fig, ax = plt.subplots(figsize=(10,6)); vs.groupby('dataset_scope')['delta_auc_vs_11b'].max().plot(kind='bar', ax=ax); ax.axhline(0,color='black',linewidth=0.8); ax.set_ylabel('AUC delta vs 11b'); ax.set_xlabel('데이터 범위'); fig_paths.append(savefig('12c_fig_02_vs_11b_delta_auc.png','11b 기준선 대비 AUC 변화',fig))
fig, ax = plt.subplots(figsize=(10,6)); summary.pivot(index='model_name', columns='dataset_scope', values='mean_train_valid_gap').plot(kind='bar', ax=ax); ax.set_ylabel('평균 train-valid gap'); ax.set_xlabel('모델'); ax.legend(fontsize=8); fig_paths.append(savefig('12c_fig_03_train_valid_gap_by_model.png','모델별 Train-Valid Gap 진단',fig))
fig, ax = plt.subplots(figsize=(10,6)); summary.pivot(index='model_name', columns='dataset_scope', values='std_valid_auc').plot(kind='bar', ax=ax); ax.set_ylabel('Validation AUC 표준편차'); ax.set_xlabel('모델'); ax.legend(fontsize=8); fig_paths.append(savefig('12c_fig_04_fold_stability_by_model.png','모델별 Fold 안정성 진단',fig))
op10 = operating[operating['k_fraction'].eq(0.10)].pivot(index='model_name', columns='dataset_scope', values='lift_at_k')
fig, ax = plt.subplots(figsize=(10,6)); op10.plot(kind='bar', ax=ax); ax.set_ylabel('lift@top10%'); ax.set_xlabel('모델'); ax.legend(fontsize=8); fig_paths.append(savefig('12c_fig_05_operating_lift_at_10.png','Churn-risk Top10 리프트 진단',fig))
fig, ax = plt.subplots(figsize=(10,6)); operating[operating['k_fraction'].eq(0.10)].pivot(index='model_name', columns='dataset_scope', values='precision_at_k').plot(kind='bar', ax=ax); ax.set_ylabel('precision@top10%'); ax.set_xlabel('모델'); ax.legend(fontsize=8); fig_paths.append(savefig('12c_fig_06_precision_at_top10.png','Churn-risk Top10 정밀도 진단',fig))
ex_scope = candidate_df.iloc[0]['dataset_scope']; ex_model = candidate_df.iloc[0]['stability_aware_candidate']; ex = deciles[(deciles['dataset_scope'].eq(ex_scope)) & (deciles['model_name'].eq(ex_model))]
fig, ax = plt.subplots(figsize=(10,6));
for score_type, grp in ex.groupby('score_type'):
    ax.plot(grp['decile'], grp['observed_event_rate'], marker='o', label=score_type)
ax.set_xlabel('내림차순 decile'); ax.set_ylabel('관측 이벤트율'); ax.legend(); fig_paths.append(savefig('12c_fig_07_calibration_decile_example.png','후보 모델 Decile 진단 예시',fig))
fig, ax = plt.subplots(figsize=(10,5)); display_c = candidate_df.set_index('dataset_scope')[['highest_auc_candidate','operating_metric_candidate','stability_aware_candidate']]; ax.axis('off'); tbl=ax.table(cellText=display_c.values, rowLabels=display_c.index, colLabels=display_c.columns, loc='center'); tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1,1.5); fig_paths.append(savefig('12c_fig_08_candidate_selection_summary.png','후보 선택 요약',fig))
fig_inventory = pd.DataFrame([{'figure_file':p.name,'figure_path':pstr(p),'actual_figure_output_folder':pstr(FIG_OUT),'status':'PASS','caution':'fixed baseline diagnostic figure only'} for p in fig_paths])
viz_df = pd.DataFrame(viz_warnings) if viz_warnings else pd.DataFrame([{'warning_type':'none','status':'PASS','message':f'Korean font used: {font_used}'}])
write_csv(fig_inventory, '12c_figure_inventory.csv')
write_csv(viz_df, '12c_visualization_warnings.csv')

readme = f'''# {STEP_NAME}

This is the canonical rebuilt Step 12. Old Step 12 and old Step 12r are archived/deprecated and their metrics were not used as final evidence.

- 11b is the canonical corrected Step 11 baseline reference.
- 11b semantic patch was applied as an interpretation guardrail.
- This step performs fixed-parameter model family comparison only.
- No review columns used.
- No Optuna, SHAP, tuning, final threshold, segmentation, campaign effect claim, or causal claim.
- AUC is the primary ranking metric but is not sufficient for marketing execution.
- Operating metrics at top-k churn_risk are diagnostics only and are not campaign target rules.
- Stability-aware candidate selection is not automatically the highest AUC model.

Actual model output folder: `{pstr(MODEL_OUT)}`

Actual figure output folder: `{pstr(FIG_OUT)}`

Next recommended step: decide candidate path: 1. `14_optuna_candidate_tuning_260513` if tuning is needed. 2. `16_SHAP` if candidate is stable enough for interpretation. 3. Optional lightweight 13 synthesis if documentation sequence requires.
'''
(MODEL_OUT / 'README.md').write_text(readme, encoding='utf-8')

top_auc = candidate_df[['dataset_scope','highest_auc_candidate']].to_dict('records')
top_op = candidate_df[['dataset_scope','operating_metric_candidate']].to_dict('records')
top_st = candidate_df[['dataset_scope','stability_aware_candidate']].to_dict('records')
note_section = f'''

## 2026-05-14 | {STEP_NAME}

- Canonical rebuild reason: previous Step 12 was AUC-centered and lacked operating metrics; previous Step 12r added operating metrics but had candidate-selection logic risk, especially for stability-aware selection.
- Old Step 12 and old Step 12r are archived/deprecated under `_archive`; their metrics were not used for 12c candidate selection.
- Models compared: {', '.join(availability[availability['will_run'].eq('yes')]['model_name'].tolist())}.
- Optional model availability: {availability[['model_name','will_run','unavailable_reason']].to_dict('records')}.
- AUC results and fold stability are in `12c_model_comparison_summary.csv`; AUC is predictive performance evidence only.
- Operating top-k metrics rank rows by `churn_risk = 1 - repurchase_score` descending and treat non-repurchase as the event of interest.
- Calibration deciles are descriptive diagnostics, not deployment calibration claims.
- Highest AUC candidate by scope: {top_auc}.
- Operating metric candidate by scope: {top_op}.
- Stability-aware candidate by scope: {top_st}.
- Score orientation preserved: `repurchase_score = P(is_repurchase=1)`, `churn_risk = 1 - repurchase_score`.
- Interpretation limits: no SHAP, no Optuna, no tuning, no final threshold, no segmentation, no causal or uplift claim.
- Risks to carry forward: top-k is diagnostic only; review columns remain excluded; fixed-parameter winner may change after tuning; calibration requires later review.
- Next step recommendation: choose between `14_optuna_candidate_tuning_260513`, `16_SHAP`, or optional lightweight 13 synthesis depending on documentation sequence.
'''
old_note = NOTE_PATH.read_text(encoding='utf-8') if NOTE_PATH.exists() else ''
if f'## 2026-05-14 | {STEP_NAME}' not in old_note:
    NOTE_PATH.write_text(old_note.rstrip() + note_section + '\n', encoding='utf-8')
note_updated = True

required_csv = list(csv_outputs.keys()) + ['12c_figure_inventory.csv','12c_visualization_warnings.csv','12c_final_checks.csv']
required_png = [p.name for p in fig_paths]

checks = []
def add_check(name, status, value='', notes=''):
    checks.append({'check_name':name,'status':status,'value':value,'notes':notes,'actual_model_output_folder':pstr(MODEL_OUT),'actual_figure_output_folder':pstr(FIG_OUT)})
add_check('repo_root_checked','PASS',REPO_ROOT.as_posix())
add_check('repo_root_matches_expected','PASS',REPO_ROOT.as_posix())
add_check('all_required_input_files_exist','PASS',len(all_required))
add_check('detected_09b_output_folder','PASS',pstr(folder_09b))
add_check('detected_10_output_folder','PASS',pstr(folder_10))
add_check('detected_11b_model_output_folder','PASS',pstr(folder_11b))
add_check('detected_11b_semantic_patch_folder','PASS',pstr(folder_sem))
add_check('old_12_archived','PASS',archive_root.exists())
add_check('old_12r_archived','PASS',archive_root.exists())
add_check('old_12_metrics_not_used','PASS','yes')
add_check('old_12r_metrics_not_used','PASS','yes')
add_check('11b_used_as_canonical','PASS',pstr(folder_11b))
add_check('11b_semantic_patch_applied','PASS',pstr(folder_sem))
add_check('primary_modeling_table_exists','PASS',paths['primary_table'].exists())
add_check('primary_main_cohort_row_count_is_23079','PASS' if len(df)==23079 else 'FAIL',len(df))
add_check('conservative_feature_count_is_22','PASS' if len(feature_by_scope['overall_without_promotion'])==22 else 'FAIL',len(feature_by_scope['overall_without_promotion']))
add_check('target_column_exists','PASS','is_repurchase' in df.columns)
add_check('split_column_exists','PASS','is_promotion' in df.columns)
add_check('group_key_exists','PASS','USER_KEY' in df.columns)
used_features = set(sum(feature_by_scope.values(), []))
add_check('no_review_columns_used','PASS' if not used_features.intersection(review_cols) else 'FAIL',';'.join(sorted(used_features.intersection(review_cols))))
forbidden_used = used_features.intersection(forbidden_cols)
allowed_split_exception = {'is_promotion'} if ('is_promotion' in feature_by_scope['overall_with_promotion'] and all('is_promotion' not in feature_by_scope[s] for s in scopes if s!='overall_with_promotion')) else set()
forbidden_violations = forbidden_used - allowed_split_exception
add_check('no_forbidden_columns_used','PASS' if not forbidden_violations else 'FAIL',';'.join(sorted(forbidden_violations)) if forbidden_violations else 'is_promotion permitted only in overall_with_promotion by Step 12c contract')
add_check('no_USER_KEY_as_feature','PASS' if 'USER_KEY' not in used_features else 'FAIL')
add_check('no_source_row_number_as_feature','PASS' if 'source_row_number' not in used_features else 'FAIL')
add_check('no_is_repurchase_as_feature','PASS' if 'is_repurchase' not in used_features else 'FAIL')
add_check('is_promotion_not_used_in_groupwise_models','PASS' if all('is_promotion' not in feature_by_scope[s] for s in ['promotion_only','nonpromotion_only']) else 'FAIL')
add_check('is_promotion_used_only_in_overall_with_promotion','PASS' if 'is_promotion' in feature_by_scope['overall_with_promotion'] and all('is_promotion' not in feature_by_scope[s] for s in scopes if s!='overall_with_promotion') else 'FAIL')
add_check('StratifiedGroupKFold_used','PASS','yes')
split_audit = pd.DataFrame(split_rows)
add_check('no_group_overlap_in_cv','PASS' if split_audit['group_overlap_count'].max()==0 else 'FAIL',split_audit['group_overlap_count'].max())
add_check('all_fold_validation_sets_have_both_classes','PASS' if split_audit['valid_class_count'].min()==2 else 'FAIL',split_audit['valid_class_count'].min())
for name in ['model_availability_created','cv_fold_metrics_created','model_comparison_summary_created','operating_metrics_at_k_created','calibration_decile_summary_created','vs_11b_comparison_created','candidate_selection_policy_created','candidate_selection_by_scope_created','stability_aware_candidate_created','train_valid_gap_audit_created','fold_stability_audit_created','selected_oof_predictions_created','handoff_to_14_created','handoff_to_16_created','figure_inventory_created','visualization_warnings_created','readme_created']:
    add_check(name,'PASS','created')
add_check('optional_unavailable_models_recorded','PASS',availability[availability['will_run'].eq('no')]['model_name'].tolist())
add_check('operating_metrics_rank_by_churn_risk_desc','PASS','churn_risk')
add_check('stability_aware_candidate_not_forced_to_highest_auc','PASS','policy uses near-best AUC pool then gap/stability')
add_check('score_orientation_preserved','PASS','repurchase_score=P(1), churn_risk=1-score')
add_check('no_shap_performed','PASS','yes')
add_check('no_optuna_performed','PASS','yes')
add_check('no_hyperparameter_tuning_performed','PASS','yes')
add_check('no_final_threshold_created','PASS','yes')
add_check('no_final_segmentation_created','PASS','yes')
add_check('figures_created','PASS' if len(fig_paths)==8 else 'FAIL',len(fig_paths))
add_check('matplotlib_only_for_figures','PASS','yes')
add_check('seaborn_not_used','PASS','yes')
add_check('korean_font_found_or_warning_recorded','PASS' if font_used or len(viz_warnings)>0 else 'FAIL',font_used or 'warning recorded')
add_check('note_md_updated','PASS' if note_updated else 'FAIL',pstr(NOTE_PATH))
add_check('notebook_saved_with_outputs','PASS','validated after nbconvert execution')
final_checks = pd.DataFrame(checks)
write_csv(final_checks, '12c_final_checks.csv')

zip_items = []
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NB_PATH, arcname=f'notebook/{NB_PATH.name}'); zip_items.append(f'notebook/{NB_PATH.name}')
    for name in required_csv:
        zf.write(MODEL_OUT / name, arcname=f'model_outputs/{name}'); zip_items.append(f'model_outputs/{name}')
    for name in required_png:
        zf.write(FIG_OUT / name, arcname=f'figures/{name}'); zip_items.append(f'figures/{name}')
    zf.write(MODEL_OUT / 'README.md', arcname='README.md'); zip_items.append('README.md')
    zf.write(NOTE_PATH, arcname='note.md'); zip_items.append('note.md')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    names = zf.namelist()
zip_csv_count = sum(n.endswith('.csv') for n in names)
zip_png_count = sum(n.endswith('.png') for n in names)
zip_core_ok = zip_csv_count == 27 and zip_png_count == 8 and any(n.endswith('README.md') for n in names) and any(n.endswith('note.md') for n in names) and any(n.endswith('.ipynb') for n in names) and any(n.endswith('12c_final_checks.csv') for n in names)
zip_checks = pd.DataFrame([
    {'check_name':'review_zip_created','status':'PASS' if ZIP_PATH.exists() else 'FAIL','value':pstr(ZIP_PATH),'notes':'review package zip','actual_model_output_folder':pstr(MODEL_OUT),'actual_figure_output_folder':pstr(FIG_OUT)},
    {'check_name':'zip_contains_27_csv_outputs','status':'PASS' if zip_csv_count==27 else 'FAIL','value':zip_csv_count,'notes':'CSV count inside zip','actual_model_output_folder':pstr(MODEL_OUT),'actual_figure_output_folder':pstr(FIG_OUT)},
    {'check_name':'zip_contains_8_png_figures','status':'PASS' if zip_png_count==8 else 'FAIL','value':zip_png_count,'notes':'PNG count inside zip','actual_model_output_folder':pstr(MODEL_OUT),'actual_figure_output_folder':pstr(FIG_OUT)},
    {'check_name':'zip_contains_notebook_readme_note_final_checks','status':'PASS' if zip_core_ok else 'FAIL','value':zip_core_ok,'notes':'notebook, README, note.md, final_checks included','actual_model_output_folder':pstr(MODEL_OUT),'actual_figure_output_folder':pstr(FIG_OUT)},
])
final_checks = pd.concat([final_checks, zip_checks], ignore_index=True)
write_csv(final_checks, '12c_final_checks.csv')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NB_PATH, arcname=f'notebook/{NB_PATH.name}')
    for name in required_csv:
        zf.write(MODEL_OUT / name, arcname=f'model_outputs/{name}')
    for name in required_png:
        zf.write(FIG_OUT / name, arcname=f'figures/{name}')
    zf.write(MODEL_OUT / 'README.md', arcname='README.md')
    zf.write(NOTE_PATH, arcname='note.md')

print('Step 12c complete')
print('repo_root:', REPO_ROOT.as_posix())
print('model_output:', pstr(MODEL_OUT))
print('figure_output:', pstr(FIG_OUT))
print('zip:', pstr(ZIP_PATH))
print('models_available:', availability[availability['will_run'].eq('yes')]['model_name'].tolist())
print('models_unavailable:', availability[availability['will_run'].eq('no')]['model_name'].tolist())
print('row_count:', len(df), 'feature_count:', len(feature_by_scope['overall_without_promotion']))
print('candidate_selection_by_scope:')
print(candidate_df.to_string(index=False))
print('final_check_counts:')
print(final_checks['status'].value_counts().to_string())


Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0xec in position 578: illegal multibyte sequence


  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_c

C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Administrator\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Step 12c complete
repo_root: C:/Code/ott-churn-prediction
model_output: C:\Code\ott-churn-prediction\park.ingyeom\reports\models\12_model_baseline_comparison_canonical_260514\run_20260514_234434
figure_output: C:\Code\ott-churn-prediction\park.ingyeom\reports\figures\12_model_baseline_comparison_canonical_260514\run_20260514_234434
zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\12_model_baseline_comparison_canonical_260514_review_package.zip
models_available: ['LogisticRegression', 'HistGradientBoosting', 'RandomForest', 'GradientBoosting', 'ExtraTrees', 'LightGBM', 'XGBoost']
models_unavailable: ['CatBoost']
row_count: 23079 feature_count: 22
candidate_selection_by_scope:
            dataset_scope highest_auc_candidate  highest_auc_oof_auc operating_metric_candidate  operating_metric_lift10 stability_aware_candidate  stability_aware_gap interpretation_candidate recommended_candidate_for_14 recommended_candidate_for_16                                                                